## Process datetime string with multiple formats to 1 format

In [2]:

import polars as pl
# Source - https://stackoverflow.com/a/76010444
# Posted by jqurious, modified by community. See post 'Timeline' for change history
# Retrieved 2026-05-02, License - CC BY-SA 4.0

df = pl.from_repr("""
┌──────────────────────────┐
│ date                     │
│ ---                      │
│ str                      │
╞══════════════════════════╡
│ Sun Jul  8 00:34:60 2001 │
│ 12Mar2022                │
│ 12/Mar/2022              │
└──────────────────────────┘
""")

fmts = "%d/%b/%Y", "%d%b%Y", "%c"

df.with_columns(
   pl.coalesce(                                          # coalesce get first not null value of all columns in one row
      pl.col("date").str.to_datetime(fmt, strict=False)
      for fmt in fmts
   )
)


date
datetime[μs]
2001-07-08 00:35:00
2022-03-12 00:00:00
2022-03-12 00:00:00


In [1]:
import io
import os
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

# Define read-only permissions for Google Drive
SCOPES = ["https://www.googleapis.com/auth/drive"]


def get_gdrive_service():
    creds = None
    # token.json stores the user's access and refresh tokens
    if os.path.exists("token.json"):
        creds = Credentials.from_authorized_user_file("token.json", SCOPES)

    # If there are no valid credentials available, let the user log in
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                r"C:\Users\ASUS\Code\data_practice\credentials.json", SCOPES
            )
            creds = flow.run_local_server(port=0)
        # Save the credentials for the next run
        with open("token.json", "w") as token:
            token.write(creds.to_json())

    return build("drive", "v3", credentials=creds)


def download_folder(service, folder_id, local_dir):
    if not os.path.exists(local_dir):
        os.makedirs(local_dir)

    # Query all files and child folders residing inside the parent folder ID
    query = f"'{folder_id}' in parents and trashed = false"
    results = (
        service.files().list(q=query, fields="files(id, name, mimeType)").execute()
    )
    items = results.get("files", [])

    for item in items:
        item_id = item["id"]
        item_name = item["name"]
        item_type = item["mimeType"]
        local_path = os.path.join(local_dir, item_name)

        if item_type == "application/vnd.google-apps.folder":
            # Recursively handle subfolders
            print(f"Entering directory: {item_name}")
            download_folder(service, item_id, local_path)
        else:
            # Handle standard files
            print(f"Downloading file: {item_name}")
            request = service.files().get_media(fileId=item_id)

            with io.FileIO(local_path, "wb") as fh:
                downloader = MediaIoBaseDownload(fh, request)
                done = False
                while not done:
                    status, done = downloader.next_chunk()
                    if status:
                        print(f"Progress: {int(status.progress() * 100)}%", end="\r")


if __name__ == "__main__":
    try:
        drive_service = get_gdrive_service()

        TARGET_FOLDER_ID = "14vCUXklVCUN_rCPKYN7ys2tDJG72SNYw"
        LOCAL_DESTINATION = "./data/test/drive"

        download_folder(drive_service, TARGET_FOLDER_ID, LOCAL_DESTINATION)

        print("\nDownload finished successfully!")

    except Exception as e:
        import traceback

        traceback.print_exc()


ModuleNotFoundError: No module named 'google'